# Stability and Reliability of XAI in ML-Based Network Intrusion Detection Systems

**Datasets:** UNSW-NB15, NSL-KDD  
**Models:** Random Forest, XGBoost, Decision Tree, Gradient Boosting  
**XAI Methods:** SHAP, LIME  
**Stability Metrics:** Cosine Similarity, Jaccard@5, Spearman Rank, Sign Agreement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Image
import os, time, joblib, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')

---
## 1. Load Datasets

In [ ]:
from preprocessing import load_dataset

datasets = {}
for name in ['UNSW-NB15', 'NSL-KDD']:
    X_tr, X_te, y_tr, y_te, sc = load_dataset(name)
    datasets[name] = (X_tr, X_te, y_tr, y_te, sc)
    print(f'{name}: {X_tr.shape[0]} train, {X_te.shape[0]} test, {X_tr.shape[1]} features')

---
## 2. Exploratory Data Analysis

In [ ]:
# UNSW-NB15 EDA
df_unsw = pd.read_csv('datasets/UNSW_NB15_training-set.csv')
print(f'UNSW-NB15 shape: {df_unsw.shape}')
print(f'Missing values: {df_unsw.isnull().sum().sum()}')
display(df_unsw.head())
display(df_unsw.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_unsw['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('UNSW-NB15: Binary Label Distribution', fontweight='bold')
axes[0].set_xticklabels(['Attack (1)', 'Normal (0)'], rotation=0)
axes[0].set_ylabel('Count')

df_unsw['attack_cat'].value_counts().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('UNSW-NB15: Attack Category Distribution', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# NSL-KDD EDA
nsl_cols = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]
df_nsl = pd.read_csv('datasets/KDDTrain+.txt', header=None, names=nsl_cols)
print(f'NSL-KDD shape: {df_nsl.shape}')
display(df_nsl.head())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_nsl['binary'] = (df_nsl['label'] != 'normal').astype(int)
df_nsl['binary'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'])
axes[0].set_title('NSL-KDD: Binary Label Distribution', fontweight='bold')
axes[0].set_xticklabels(['Attack','Normal'], rotation=0)

df_nsl['label'].value_counts().head(10).plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('NSL-KDD: Top 10 Attack Types', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, df, title in [(axes[0], df_unsw, 'UNSW-NB15'), (axes[1], df_nsl, 'NSL-KDD')]:
    num = df.select_dtypes(include=[np.number])
    corr = num.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, ax=ax,
                square=True, linewidths=0.3, cbar_kws={'shrink': 0.6})
    ax.set_title(f'{title}: Correlation Heatmap', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# UNSW-NB15 Feature Distributions
top_feats = ['dur','sbytes','dbytes','sttl','dttl','sload','dload','smean','dmean','ct_srv_src']
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, feat in zip(axes.flatten(), top_feats):
    if feat in df_unsw.columns:
        for label, color in [(0,'steelblue'),(1,'coral')]:
            data = df_unsw[df_unsw['label']==label][feat]
            data = data[data < data.quantile(0.95)]
            ax.hist(data, bins=30, alpha=0.6, color=color,
                    label='Normal' if label==0 else 'Attack')
        ax.set_title(feat, fontsize=10)
        ax.legend(fontsize=7)
plt.suptitle('UNSW-NB15: Feature Distributions by Label', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# NSL-KDD Feature Distributions
nsl_numeric_feats = ['duration','src_bytes','dst_bytes','hot','count','srv_count',
                     'dst_host_count','dst_host_srv_count','serror_rate','rerror_rate']
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, feat in zip(axes.flatten(), nsl_numeric_feats):
    if feat in df_nsl.columns:
        for label, color in [(0,'steelblue'),(1,'coral')]:
            data = df_nsl[df_nsl['binary']==label][feat]
            data = data[data < data.quantile(0.95)]
            ax.hist(data, bins=30, alpha=0.6, color=color,
                    label='Normal' if label==0 else 'Attack')
        ax.set_title(feat, fontsize=10)
        ax.legend(fontsize=7)
plt.suptitle('NSL-KDD: Feature Distributions by Label', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Model Training & Performance

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

MODEL_DEFS = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, max_depth=20),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5),
}

results_all = []
trained_models = {}

for ds_name in ['UNSW-NB15', 'NSL-KDD']:
    X_train, X_test, y_train, y_test, _ = datasets[ds_name]
    for model_name, model in MODEL_DEFS.items():
        # Load pre-trained if available, else train
        mpath = f'models/{ds_name}_{model_name}.pkl'
        if os.path.exists(mpath):
            model = joblib.load(mpath)
            train_time = 0
        else:
            t0 = time.time()
            model.fit(X_train, y_train)
            train_time = time.time() - t0
            joblib.dump(model, mpath)

        trained_models[(ds_name, model_name)] = model
        preds = model.predict(X_test)
        results_all.append({
            'Dataset': ds_name, 'Model': model_name,
            'Accuracy': round(accuracy_score(y_test, preds), 4),
            'Precision': round(precision_score(y_test, preds, zero_division=0), 4),
            'Recall': round(recall_score(y_test, preds, zero_division=0), 4),
            'F1-Score': round(f1_score(y_test, preds, zero_division=0), 4),
        })

perf = pd.DataFrame(results_all)
display(HTML('<h3>Model Performance Table</h3>'))
display(perf.style.background_gradient(cmap='YlGnBu', subset=['Accuracy','Precision','Recall','F1-Score']))

In [ ]:
# Performance Bar Charts
PALETTE = sns.color_palette('Set2', 4)
MODEL_ORDER = ['RandomForest','XGBoost','DecisionTree','GradientBoosting']
metrics = ['Accuracy','Precision','Recall','F1-Score']

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharey=True)
for ax, ds in zip(axes, ['UNSW-NB15','NSL-KDD']):
    subset = perf[perf['Dataset']==ds].set_index('Model').loc[MODEL_ORDER]
    x = np.arange(len(MODEL_ORDER))
    w = 0.18
    for i, m in enumerate(metrics):
        bars = ax.bar(x + i*w, subset[m], w, label=m, color=PALETTE[i])
        for j, v in enumerate(subset[m]):
            ax.text(x[j]+i*w, v+0.005, f'{v:.3f}', ha='center', fontsize=7, rotation=45)
    ax.set_xticks(x + 1.5*w)
    ax.set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
    ax.set_ylim(0.5, 1.05)
    ax.set_title(ds, fontsize=14, fontweight='bold')
    ax.legend(fontsize=8, loc='lower left')
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# F1-Score Heatmap
fig, ax = plt.subplots(figsize=(8, 4))
pivot_f1 = perf.pivot(index='Model', columns='Dataset', values='F1-Score').loc[MODEL_ORDER]
sns.heatmap(pivot_f1, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax,
            linewidths=1, cbar_kws={'label':'F1-Score'}, vmin=0.7, vmax=1.0)
ax.set_title('F1-Score Heatmap: Model x Dataset', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Training Time Comparison
perf_with_time = pd.read_csv('results/model_performance.csv')
fig, ax = plt.subplots(figsize=(10, 5))
pivot_time = perf_with_time.pivot(index='Model', columns='Dataset', values='Train Time (s)')
pivot_time = pivot_time.loc[MODEL_ORDER]
pivot_time.plot(kind='bar', ax=ax, color=['steelblue','coral'], edgecolor='black', linewidth=0.5)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1fs', fontsize=8, padding=2)
ax.set_ylabel('Training Time (seconds)')
ax.set_title('Training Time: All Models x All Datasets', fontweight='bold', fontsize=13)
ax.set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
ax.legend(title='Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# All Metrics Heatmaps (Accuracy, Precision, Recall, F1)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, metric in zip(axes, ['Accuracy','Precision','Recall','F1-Score']):
    pivot = perf.pivot(index='Model', columns='Dataset', values=metric).loc[MODEL_ORDER]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax,
                linewidths=1, vmin=0.5, vmax=1.0,
                cbar_kws={'label': metric})
    ax.set_title(metric, fontweight='bold', fontsize=12)
    if metric != 'Accuracy':
        ax.set_ylabel('')
fig.suptitle('All Performance Metrics: Model x Dataset', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices (all 8 in a grid)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for row, ds in enumerate(['UNSW-NB15','NSL-KDD']):
    X_tr, X_te, y_tr, y_te, _ = datasets[ds]
    for col, mn in enumerate(MODEL_ORDER):
        ax = axes[row][col]
        model = trained_models[(ds, mn)]
        preds = model.predict(X_te)
        cm = confusion_matrix(y_te, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal','Attack'], yticklabels=['Normal','Attack'])
        ax.set_title(f'{mn}\n{ds}', fontsize=10, fontweight='bold')
        ax.set_ylabel('Actual' if col==0 else '')
        ax.set_xlabel('Predicted')

fig.suptitle('Confusion Matrices: All Models x All Datasets', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Reports for All Models
for ds_name in ['UNSW-NB15','NSL-KDD']:
    X_tr, X_te, y_tr, y_te, _ = datasets[ds_name]
    for mn in MODEL_ORDER:
        model = trained_models[(ds_name, mn)]
        preds = model.predict(X_te)
        print(f'\n{"="*50}')
        print(f'  {mn} on {ds_name}')
        print(f'{"="*50}')
        print(classification_report(y_te, preds, target_names=['Normal','Attack']))

---
## 4. XAI Analysis (SHAP & LIME)

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import spearmanr

SAMPLE_SIZE = 200
TOP_K = 10
agreement_results = []

for ds_name in ['UNSW-NB15','NSL-KDD']:
    X_train, X_test, y_train, y_test, _ = datasets[ds_name]
    feature_names = X_train.columns.tolist()
    X_sample = X_test.sample(min(SAMPLE_SIZE, len(X_test)), random_state=42)

    for model_name in MODEL_ORDER:
        model = trained_models[(ds_name, model_name)]
        print(f'\n--- {model_name} on {ds_name} ---')

        # SHAP
        t0 = time.time()
        explainer = shap.TreeExplainer(model)
        shap_values = np.array(explainer.shap_values(X_sample))
        if shap_values.ndim == 3:
            shap_vals = shap_values[:, :, 1]
        else:
            shap_vals = shap_values
        shap_time = time.time() - t0

        shap_importance = np.abs(shap_vals).mean(axis=0).flatten()
        shap_top = [feature_names[int(i)] for i in np.argsort(shap_importance)[::-1][:TOP_K]]

        # LIME
        t0 = time.time()
        lime_exp = LimeTabularExplainer(X_train.values, feature_names=feature_names,
                                        class_names=['Normal','Attack'], mode='classification')
        lime_importance = np.zeros(len(feature_names))
        n_lime = min(50, len(X_sample))
        for i in range(n_lime):
            exp = lime_exp.explain_instance(
                X_sample.iloc[i].values,
                lambda x: model.predict_proba(pd.DataFrame(x, columns=feature_names)),
                num_features=TOP_K)
            for feat_str, weight in exp.as_list():
                for fi, fn in enumerate(feature_names):
                    if fn in feat_str:
                        lime_importance[fi] += abs(weight)
        lime_importance /= n_lime
        lime_time = time.time() - t0
        lime_top = [feature_names[int(i)] for i in np.argsort(lime_importance)[::-1][:TOP_K]]

        overlap = len(set(shap_top) & set(lime_top))
        jaccard = overlap / len(set(shap_top) | set(lime_top))
        sp_corr, _ = spearmanr(
            np.argsort(np.argsort(-shap_importance.flatten())),
            np.argsort(np.argsort(-lime_importance.flatten()))
        )
        agreement_results.append({
            'Dataset': ds_name, 'Model': model_name,
            'Top-10 Overlap': overlap, 'Jaccard': round(jaccard,4),
            'Spearman': round(sp_corr,4),
            'SHAP Time (s)': round(shap_time,2),
            'LIME Time (s)': round(lime_time,2),
            '_shap_imp': shap_importance, '_lime_imp': lime_importance,
            '_shap_vals': shap_vals, '_X_sample': X_sample,
        })
        print(f'  SHAP: {shap_time:.1f}s | LIME: {lime_time:.1f}s | Overlap: {overlap}/{TOP_K}')

agree_df = pd.DataFrame([{k:v for k,v in r.items() if not k.startswith('_')} for r in agreement_results])
display(HTML('<h3>SHAP vs LIME Feature Agreement</h3>'))
display(agree_df)

In [ ]:
# SHAP Summary Plots (all 8 in a grid)
fig, axes = plt.subplots(2, 4, figsize=(28, 12))
for row, ds in enumerate(['UNSW-NB15','NSL-KDD']):
    for col, mn in enumerate(MODEL_ORDER):
        ax = axes[row][col]
        r = [r for r in agreement_results if r['Dataset']==ds and r['Model']==mn][0]
        plt.sca(ax)
        shap.summary_plot(r['_shap_vals'], r['_X_sample'], plot_type='dot',
                          max_display=10, show=False)
        ax.set_title(f'{mn}\n{ds}', fontsize=10, fontweight='bold')

fig.suptitle('SHAP Feature Importance — All Models x Datasets', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP vs LIME Feature Agreement Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in zip(axes,
    ['Top-10 Overlap','Jaccard','Spearman'],
    ['Top-10 Feature Overlap','Jaccard Similarity','Spearman Rank Correlation']):
    pivot = agree_df.pivot(index='Model', columns='Dataset', values=metric).loc[MODEL_ORDER]
    pivot.plot(kind='bar', ax=ax, color=['steelblue','coral'], edgecolor='black', linewidth=0.5)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', fontsize=7, padding=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(MODEL_ORDER, rotation=20, ha='right', fontsize=9)
    ax.legend(title='Dataset', fontsize=8)

fig.suptitle('SHAP vs LIME Feature Agreement', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side Feature Importance: SHAP vs LIME (all 8)
fig, axes = plt.subplots(2, 4, figsize=(28, 10))
for row, ds in enumerate(['UNSW-NB15','NSL-KDD']):
    for col, mn in enumerate(MODEL_ORDER):
        ax = axes[row][col]
        r = [r for r in agreement_results if r['Dataset']==ds and r['Model']==mn][0]
        fnames = datasets[ds][0].columns.tolist()
        si, li = r['_shap_imp'], r['_lime_imp']
        top_idx = list(set(
            list(np.argsort(si)[::-1][:7]) + list(np.argsort(li)[::-1][:7])
        ))[:10]
        names = [fnames[int(i)] for i in top_idx]
        x = np.arange(len(names))
        w = 0.35
        ax.barh(x - w/2, [si[i] for i in top_idx], w, label='SHAP', color='steelblue')
        ax.barh(x + w/2, [li[i] for i in top_idx], w, label='LIME', color='coral')
        ax.set_yticks(x)
        ax.set_yticklabels(names, fontsize=7)
        ax.set_title(f'{mn} — {ds}', fontsize=9, fontweight='bold')
        ax.legend(fontsize=7)

fig.suptitle('SHAP vs LIME Feature Importance (All Models x Datasets)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# XAI Runtime Comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(MODEL_ORDER))
w = 0.18
colors = ['#2196F3','#64B5F6','#F44336','#EF9A9A']
for i, (ds, method, col) in enumerate([
    ('UNSW-NB15','SHAP','SHAP Time (s)'), ('UNSW-NB15','LIME','LIME Time (s)'),
    ('NSL-KDD','SHAP','SHAP Time (s)'), ('NSL-KDD','LIME','LIME Time (s)')
]):
    subset = agree_df[agree_df['Dataset']==ds].set_index('Model').loc[MODEL_ORDER]
    ax.bar(x + i*w, subset[col], w, label=f'{method} ({ds.split("-")[0]})', color=colors[i])
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
ax.set_ylabel('Time (seconds)')
ax.set_title('XAI Computation Time Comparison', fontweight='bold', fontsize=13)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 5. Stability Analysis

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

NOISE_LEVELS = [0.01, 0.02, 0.05, 0.10]
N_SHAP = 50
N_LIME = 15
TOP_K_STAB = 5

def compute_stability(v1, v2):
    cos = cosine_similarity(v1.reshape(1,-1), v2.reshape(1,-1))[0][0] if np.linalg.norm(v1)>0 and np.linalg.norm(v2)>0 else 1.0
    t1, t2 = set(np.argsort(np.abs(v1))[::-1][:TOP_K_STAB]), set(np.argsort(np.abs(v2))[::-1][:TOP_K_STAB])
    jac = len(t1 & t2) / max(len(t1 | t2), 1)
    sp = spearmanr(v1, v2)[0] if np.std(v1)>0 and np.std(v2)>0 else 1.0
    s1, s2 = np.sign(v1), np.sign(v2)
    nz = (s1!=0)|(s2!=0)
    sign = (s1[nz]==s2[nz]).mean() if nz.sum()>0 else 1.0
    return cos, jac, sp, sign

def lime_vec(exp, fnames):
    v = np.zeros(len(fnames))
    for s, w in exp.as_list():
        for i, fn in enumerate(fnames):
            if fn in s: v[i] = w
    return v

def extract_class1(sv):
    sv = np.array(sv)
    return sv[:,:,1] if sv.ndim==3 else sv

stab_results = []

for ds_name in ['UNSW-NB15','NSL-KDD']:
    X_train, X_test, y_train, y_test, _ = datasets[ds_name]
    fnames = X_train.columns.tolist()

    for mn in MODEL_ORDER:
        model = trained_models[(ds_name, mn)]
        shap_exp = shap.TreeExplainer(model)
        lime_exp = LimeTabularExplainer(X_train.values[:2000], feature_names=fnames,
                                        class_names=['Normal','Attack'], mode='classification')
        X_s = X_test.sample(N_SHAP, random_state=42)

        for noise in NOISE_LEVELS:
            X_p = pd.DataFrame(X_s.values + np.random.normal(0, noise, X_s.shape),
                               columns=fnames, index=X_s.index)
            sv_o = extract_class1(shap_exp.shap_values(X_s))
            sv_p = extract_class1(shap_exp.shap_values(X_p))
            shap_m = [compute_stability(sv_o[i], sv_p[i]) for i in range(N_SHAP)]

            lime_m = []
            for i in range(N_LIME):
                eo = lime_exp.explain_instance(X_s.iloc[i].values,
                    lambda x: model.predict_proba(pd.DataFrame(x, columns=fnames)), num_features=10)
                ep = lime_exp.explain_instance(X_p.iloc[i].values,
                    lambda x: model.predict_proba(pd.DataFrame(x, columns=fnames)), num_features=10)
                lime_m.append(compute_stability(lime_vec(eo, fnames), lime_vec(ep, fnames)))

            for xai, ms in [('SHAP', shap_m), ('LIME', lime_m)]:
                arr = np.array(ms)
                stab_results.append({
                    'Dataset': ds_name, 'Model': mn, 'XAI': xai, 'Noise': noise,
                    'Cosine': round(arr[:,0].mean(),4), 'Jaccard@5': round(arr[:,1].mean(),4),
                    'Spearman': round(arr[:,2].mean(),4), 'Sign Agree': round(arr[:,3].mean(),4)
                })

            print(f'{mn} {ds_name} noise={noise:.2f} done')

stab_df = pd.DataFrame(stab_results)
display(HTML('<h3>Stability Results</h3>'))
display(stab_df)

In [ ]:
# Stability vs Noise (all models, both datasets, SHAP vs LIME)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for idx, (ds, xai) in enumerate([('UNSW-NB15','SHAP'),('UNSW-NB15','LIME'),
                                   ('NSL-KDD','SHAP'),('NSL-KDD','LIME')]):
    ax = axes[idx//2][idx%2]
    sub = stab_df[(stab_df['Dataset']==ds)&(stab_df['XAI']==xai)]
    for mi, mn in enumerate(MODEL_ORDER):
        d = sub[sub['Model']==mn]
        if len(d)>0:
            ax.plot(d['Noise'], d['Cosine'], marker='o', label=mn,
                    color=PALETTE[mi], linewidth=2, markersize=7)
    ax.set_xlabel('Noise Level')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title(f'{xai} Stability — {ds}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.2, 1.05)
    ax.grid(True, alpha=0.3)

fig.suptitle('Explanation Stability vs Perturbation Noise', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Stability Heatmaps (SHAP vs LIME at noise=0.02)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, xai in zip(axes, ['SHAP','LIME']):
    sub = stab_df[(stab_df['Noise']==0.02)&(stab_df['XAI']==xai)]
    pivot = sub.pivot(index='Model', columns='Dataset', values='Cosine').loc[MODEL_ORDER]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax,
                linewidths=1, vmin=-0.2, vmax=1.0,
                cbar_kws={'label':'Cosine Similarity'})
    ax.set_title(f'{xai} Stability (noise=0.02)', fontweight='bold')

fig.suptitle('Explanation Stability Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Multi-metric stability comparison at noise=0.02
stab_met = ['Cosine','Jaccard@5','Spearman','Sign Agree']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, ds in zip(axes, ['UNSW-NB15','NSL-KDD']):
    sub = stab_df[(stab_df['Dataset']==ds)&(stab_df['Noise']==0.02)]
    x = np.arange(len(stab_met))
    w, off = 0.1, 0
    for mi, mn in enumerate(MODEL_ORDER):
        for xi, xai in enumerate(['SHAP','LIME']):
            row = sub[(sub['Model']==mn)&(sub['XAI']==xai)]
            if len(row)==0: continue
            vals = [row[m].values[0] for m in stab_met]
            ax.bar(x + off*w, vals, w, label=f'{mn[:6]} {xai}',
                   color=PALETTE[mi], alpha=1.0 if xai=='SHAP' else 0.5,
                   edgecolor='black', linewidth=0.3)
            off += 1
    ax.set_xticks(x + 3.5*w)
    ax.set_xticklabels(stab_met)
    ax.set_title(f'{ds} — All Stability Metrics (noise=0.02)', fontweight='bold')
    ax.set_ylim(-0.3, 1.1)
    ax.legend(fontsize=5.5, ncol=2, loc='upper right')
    ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')

fig.suptitle('Stability Metrics: SHAP vs LIME (All Models)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Stability vs Noise — All 4 Metrics (Jaccard, Spearman, Sign Agree alongside Cosine)
stab_met_all = ['Cosine','Jaccard@5','Spearman','Sign Agree']
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
for row, ds in enumerate(['UNSW-NB15','NSL-KDD']):
    for col, metric in enumerate(stab_met_all):
        ax = axes[row][col]
        for xai, ls in [('SHAP','-'),('LIME','--')]:
            sub = stab_df[(stab_df['Dataset']==ds)&(stab_df['XAI']==xai)]
            for mi, mn in enumerate(MODEL_ORDER):
                d = sub[sub['Model']==mn]
                if len(d)>0:
                    label = f'{mn[:6]} {xai}' if col==0 else None
                    ax.plot(d['Noise'], d[metric], marker='o' if xai=='SHAP' else 's',
                            linestyle=ls, color=PALETTE[mi], linewidth=1.5,
                            markersize=5, label=label, alpha=1.0 if xai=='SHAP' else 0.6)
        ax.set_xlabel('Noise Level')
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} — {ds}', fontsize=10, fontweight='bold')
        ax.set_ylim(-0.3, 1.1)
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='black', linewidth=0.3, linestyle=':')
        if col == 0:
            ax.legend(fontsize=6, ncol=1, loc='lower left')

fig.suptitle('All Stability Metrics vs Noise Level (Solid=SHAP, Dashed=LIME)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Grand SHAP vs LIME Average Stability (across all models, noise=0.02)
avg_stab = stab_df[stab_df['Noise']==0.02].groupby(['Dataset','XAI'])[['Cosine','Jaccard@5','Spearman','Sign Agree']].mean()
avg_stab = avg_stab.reset_index()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
stab_met_all = ['Cosine','Jaccard@5','Spearman','Sign Agree']
for ax, metric in zip(axes, stab_met_all):
    pivot = avg_stab.pivot(index='Dataset', columns='XAI', values=metric)
    pivot[['SHAP','LIME']].plot(kind='bar', ax=ax, color=['steelblue','coral'],
                                  edgecolor='black', linewidth=0.5)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.3f', fontsize=8, padding=2)
    ax.set_title(metric, fontweight='bold', fontsize=12)
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.set_ylim(-0.1, 1.1)
    ax.axhline(y=0, color='black', linewidth=0.3, linestyle=':')
    ax.legend(fontsize=8)

fig.suptitle('Average Stability: SHAP vs LIME (across all models, noise=0.02)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correct vs Incorrect Prediction Stability
print('\n=== Correct vs Incorrect Prediction Stability (noise=0.02) ===\n')
ci_results = []
for ds_name in ['UNSW-NB15','NSL-KDD']:
    X_train, X_test, y_train, y_test, _ = datasets[ds_name]
    fnames = X_train.columns.tolist()
    for mn in MODEL_ORDER:
        model = trained_models[(ds_name, mn)]
        shap_exp = shap.TreeExplainer(model)
        preds = model.predict(X_test)
        for group, idx in [('Correct', np.where(preds==y_test.values)[0]),
                           ('Incorrect', np.where(preds!=y_test.values)[0])]:
            n = min(30, len(idx))
            if n == 0: continue
            Xg = X_test.iloc[idx].sample(n, random_state=42)
            Xp = pd.DataFrame(Xg.values + np.random.normal(0, 0.02, Xg.shape),
                              columns=fnames, index=Xg.index)
            so = extract_class1(shap_exp.shap_values(Xg))
            sp = extract_class1(shap_exp.shap_values(Xp))
            cos_vals = [cosine_similarity(so[i:i+1], sp[i:i+1])[0][0] for i in range(n)]
            ci_results.append({'Dataset':ds_name,'Model':mn,'Group':group,
                               'SHAP Cosine (mean)':round(np.mean(cos_vals),4),
                               'SHAP Cosine (std)':round(np.std(cos_vals),4)})

ci_df = pd.DataFrame(ci_results)
display(ci_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ds in zip(axes, ['UNSW-NB15','NSL-KDD']):
    sub = ci_df[ci_df['Dataset']==ds]
    pivot = sub.pivot(index='Model', columns='Group', values='SHAP Cosine (mean)').loc[MODEL_ORDER]
    pivot.plot(kind='bar', ax=ax, color=['#4CAF50','#FF5722'], edgecolor='black', linewidth=0.5)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.3f', fontsize=8, padding=2)
    ax.set_title(f'{ds}: Correct vs Incorrect Stability', fontweight='bold')
    ax.set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
    ax.set_ylim(0, 1.05)
    ax.legend(title='Prediction')

fig.suptitle('SHAP Stability: Correct vs Incorrect Predictions (noise=0.02)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Summary

In [ ]:
display(HTML('<h2>Final Summary Tables</h2>'))

display(HTML('<h3>1. Model Performance</h3>'))
display(perf.style.background_gradient(cmap='YlGnBu', subset=['Accuracy','Precision','Recall','F1-Score']))

display(HTML('<h3>2. SHAP vs LIME Feature Agreement</h3>'))
display(agree_df)

display(HTML('<h3>3. Stability at noise=0.02</h3>'))
display(stab_df[stab_df['Noise']==0.02].reset_index(drop=True))

display(HTML('<h3>4. Correct vs Incorrect Stability</h3>'))
display(ci_df)

print('\n=== All analysis complete ===')